In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hexgrid import HexCoord, HexGrid, build_obstacle_map, get_neighbors_at_radius, VelocityState, image_to_hex_obstacles, hexes_in_between_solid
from hexstar import HStarProblem, HStarSearch, h_cost_travel_time, construct_full_solution, HStarProblem, BidHStarSearch
from hexplot import plot_hex_grid_2, plot_velocity_profile

import random

In [3]:
import time
from collections.abc import Mapping

def format_cache_key(cache_key):
    (
        seed,
        radius,
        a_min,
        a_max,
        ay_window_ms,
        jps_horizon,
        n_step,
        join_tolerance,
        state_key_mode,
        search_type,
    ) = cache_key

    return (
        f"s{seed}"
        f"_r{radius}"
        f"_amin{a_min}"
        f"_amax{a_max}"
        f"_ay{ay_window_ms}"
        f"_jps{'N' if jps_horizon is None else jps_horizon}"
        f"_n{n_step}"
        f"_jt{join_tolerance}"
        f"_{state_key_mode}"
        f"_{search_type}"
    )

def flatten_dict(d, prefix=""):
    flat = {}

    if d is None:
        return flat

    for k, v in d.items():
        key = f"{prefix}_{k}" if prefix else str(k)

        if isinstance(v, Mapping):
            flat.update(flatten_dict(v, key))
        else:
            flat[key] = v

    return flat
def run_trial_bi(
    map_data,
    a_min,
    a_max,
    ay_window_ms,
    jps_horizon,
    n_step,
    solution_cache=None,
    join_tolerance=50,
    state_key_mode="location",
    progress_every=1,
    progress_top_k=3,
):
    cache_key = (
        map_data["seed"],
        map_data["radius"],
        a_min,
        a_max,
        ay_window_ms,
        jps_horizon,
        n_step,
        join_tolerance,
        state_key_mode,
        "bidirectional",
    )
    solution_filename = (
        f"s{map_data['seed']}"
        f"_r{map_data['radius']}"
        f"_d{a_min}"
        f"_a{a_max}"
        f"_y{ay_window_ms}"
        f"_j{'N' if jps_horizon is None else jps_horizon}"
        f"_n{n_step}"
        f"_t{join_tolerance}"
        f"_l"
        f"_bi.jpg"
    )

    start_time = time.perf_counter()

    hg = None
    problem = None
    bi_s = None
    bi_solution = None
    bi_solution_path = None
    benchmarks_bi = None

    try:
        hg = HexGrid(
            map_data.get("hex_size", 1),
            map_data["obstacles"]
        )

        problem = HStarProblem(
            hg,
            map_data["start"],
            map_data["goal"],
            a_max,
            a_min,
            collision_radius=0,
            jps_horizon=jps_horizon,
            ay_window_ms=ay_window_ms,
            n_step=n_step,
            start_v=VelocityState(0, None),
            goal_v=VelocityState(0, None)
        )

        bi_s = BidHStarSearch(
            problem=problem,
            heuristic=h_cost_travel_time,
            join_tolerance=join_tolerance,
            enable_benchmarking=True,
            state_key_mode=state_key_mode,
            progress_every=progress_every,
            progress_top_k=progress_top_k,
        )

        bi_solution = bi_s.search()

        runtime_ms = (time.perf_counter() - start_time) * 1000

        benchmarks_bi = bi_s.get_benchmarks()

        if bi_solution is None:
            success = False
            bi_solution_path = None
            path_length = None
        else:
            success = True
            bi_solution_path = construct_full_solution(bi_solution)
            path_length = len(bi_solution_path)

        result = {
            "cache_key": cache_key,
            "solution_filename": solution_filename,
            "search_type": "bidirectional",

            "success": success,
            #"runtime_ms_outer": runtime_ms,
            "path_length": path_length,

            "a_min": a_min,
            "a_max": a_max,
            "ay_window_ms": ay_window_ms,
            "jps_horizon": jps_horizon,
            "n_step": n_step,

            "join_tolerance": join_tolerance,
            "state_key_mode": state_key_mode,
            #"progress_every": progress_every,
            #"progress_top_k": progress_top_k,

            "seed": map_data.get("seed"),
            "radius": map_data.get("radius"),
            "obs_density": map_data.get("obs_density"),
            "num_obstacles": len(map_data.get("obstacles", [])),
            "start": map_data.get("start"),
            "goal": map_data.get("goal"),

            "error": None,
        }

        # Flatten benchmark metrics into result row.
        for k, v in flatten_dict(benchmarks_bi, prefix="bi").items():
            result[k] = v

        if solution_cache is not None:
            solution_cache[cache_key] = {
                "cache_key": cache_key,
                "success": success,
                "error": None,
                "search": bi_s,
                "problem": problem,
                "solution": bi_solution,
                "solution_path": bi_solution_path,
                "benchmarks": benchmarks_bi,
                "map_data": map_data,
                "result": result,
            }

        return result

    except Exception as e:
        runtime_ms = (time.perf_counter() - start_time) * 1000
        error_msg = str(e)

        # Try to still capture benchmarks if the search object exists.
        if bi_s is not None:
            try:
                benchmarks_bi = bi_s.get_benchmarks()
            except Exception:
                benchmarks_bi = None

        result = {
            "cache_key": cache_key,
            "solution_filename": solution_filename,
            "search_type": "bidirectional",

            "success": success,
            #"runtime_ms_outer": runtime_ms,
            "path_length": path_length,

            "a_min": a_min,
            "a_max": a_max,
            "ay_window_ms": ay_window_ms,
            "jps_horizon": jps_horizon,
            "n_step": n_step,

            "join_tolerance": join_tolerance,
            "state_key_mode": state_key_mode,
            #"progress_every": progress_every,
            #"progress_top_k": progress_top_k,

            "seed": map_data.get("seed"),
            "radius": map_data.get("radius"),
            "obs_density": map_data.get("obs_density"),
            "num_obstacles": len(map_data.get("obstacles", [])),
            "start": map_data.get("start"),
            "goal": map_data.get("goal"),

            "error": error_msg,
        }

        # Include any partial benchmark data that exists.
        for k, v in flatten_dict(benchmarks_bi, prefix="bi").items():
            result[k] = v

        if solution_cache is not None:
            solution_cache[cache_key] = {
                "cache_key": cache_key,
                "success": False,
                "error": error_msg,
                "search": bi_s,
                "problem": problem,
                "solution": bi_solution,
                "solution_path": bi_solution_path,
                "benchmarks": benchmarks_bi,
                "map_data": map_data,
                "result": result,
            }

        return result

In [ ]:
def run_trial_from_ck(cache_key):
    

In [ ]:
def run_trial_bi(
    map_data,
    a_min,
    a_max,
    ay_window_ms,
    jps_horizon,
    n_step,
    solution_cache=None,
    join_tolerance=50,
    state_key_mode="location",
    progress_every=1,
    progress_top_k=3,
):

In [4]:
ck = (0, 15, 2, 1, 3, 1, 1, 50, 'location', 'bidirectional')
result = run_trial_bi(ck)
result

TypeError: run_trial_bi() missing 5 required positional arguments: 'a_min', 'a_max', 'ay_window_ms', 'jps_horizon', and 'n_step'